In [1]:
import pandas as pd
from io import StringIO
import gzip

file_path = "20260711_20260711_Rexburg_Idaho.siz"  # <-- change to your .siz file name

def read_aeronet_siz(path: str) -> pd.DataFrame:
    # 1) Read raw text (handles plain or gzipped)
    if path.lower().endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
            text = f.read()
    else:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            text = f.read()

    # 2) Find the real CSV header line (starts with "Site,Date")
    lines = text.splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.startswith("Site,Date"):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Couldn't find the header line starting with 'Site,Date'.")

    # 3) Keep only header + data lines
    csv_text = "\n".join(lines[header_idx:])

    # 4) Load into DataFrame
    df = pd.read_csv(StringIO(csv_text))

    # 5) Clean columns a bit (optional)
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(" ", "_")
    )

    # 6) Convert what we can to numbers
    df = df.apply(pd.to_numeric, errors="ignore")

    # 7) Make a datetime column (super useful)
    date_col = "Date(dd:mm:yyyy)"
    time_col = "Time(hh:mm:ss)"
    if date_col in df.columns and time_col in df.columns:
        df["Datetime"] = pd.to_datetime(
            df[date_col].astype(str) + " " + df[time_col].astype(str),
            format="%d:%m:%Y %H:%M:%S",
            errors="coerce"
        )

    return df

df = read_aeronet_siz(file_path)

print(df.shape)
print(df.head(3))

# Optional: save to CSV
df.to_csv("20260711_Rexburg_Idaho.csv", index=False)

(6, 64)
            Site Date(dd:mm:yyyy) Time(hh:mm:ss)  Day_of_Year  \
0  Rexburg_Idaho       11:07:2026       13:31:45          192   
1  Rexburg_Idaho       11:07:2026       14:08:53          192   
2  Rexburg_Idaho       11:07:2026       15:07:11          192   

   Day_of_Year(Fraction)  0.050000  0.065604  0.086077  0.112939  0.148184  \
0             192.563715  0.000088  0.000476  0.001610  0.003467  0.004919   
1             192.589502  0.000095  0.000564  0.001892  0.003650  0.004311   
2             192.629988  0.000137  0.001029  0.003302  0.004904  0.004150   

   ...  If_AOD_is_L2  Last_Processing_Date(dd:mm:yyyy)  \
0  ...             0                        13:07:2026   
1  ...             0                        13:07:2026   
2  ...             0                        13:07:2026   

   Last_Processing_Time(hh:mm:ss)  Instrument_Number  Latitude(Degrees)  \
0                        05:01:21               1468           43.82011   
1                        05:05:30  

C:\Users\kaede\AppData\Local\Temp\ipykernel_31480\4112324928.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors="ignore")
